In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

import numpy as np
import pandas as pd

def near_psd_corr(M, eps=0.0):
    """Return a near-PSD correlation matrix from a covariance/correlation input."""
    is_df = isinstance(M, pd.DataFrame)
    A = M.to_numpy(float) if is_df else np.array(M, float)

    # symmetrize
    A = 0.5 * (A + A.T)

    # if covariance -> make it correlation
    d = np.diag(A)
    if not np.allclose(d, 1.0):
        sd = np.sqrt(np.maximum(d, 1e-18))
        A = (A / sd[:, None]) / sd[None, :]

    # eigenvalue clip (PSD) + unit diagonal
    w, V = np.linalg.eigh(A)
    w = np.clip(w, eps, None)
    A_psd = (V * w) @ V.T
    np.fill_diagonal(A_psd, 1.0)
    A_psd = 0.5 * (A_psd + A_psd.T)

    return pd.DataFrame(A_psd, index=M.index, columns=M.columns) if is_df else A_psd

# Read data and give output
DATA_DIR = Path.cwd() / "testfiles_" / "data"
csv_path = DATA_DIR / "testout_1.4.csv"

# File has header row and no index; set row labels = column labels
M = pd.read_csv(csv_path)
for c in M.columns:
    M[c] = pd.to_numeric(M[c], errors="coerce")
if M.shape[0] != M.shape[1]:
    raise ValueError(f"Matrix must be square, got {M.shape}")
M.index = M.columns

near_corr = near_psd_corr(M)
print(near_corr)

          x1        x2        x3        x4        x5
x1  1.000000 -0.483199 -0.241787 -0.067767 -0.714761
x2 -0.483199  1.000000  0.015446  0.405660  0.178286
x3 -0.241787  0.015446  1.000000  0.488250  0.336248
x4 -0.067767  0.405660  0.488250  1.000000 -0.322136
x5 -0.714761  0.178286  0.336248 -0.322136  1.000000
